FastText CC is the model of choice since its trained on generelized data instead of only tweets it got  articales blogs social media etc...


In [1]:
from pathlib import Path
import pandas as pd

# 1. Chargement des données nettoyées
DATA_DIR = Path(r"c:\Users\user\Projets_Ghoufrane\LLMS\Decetction-Persuation\data")
df_train = pd.read_json(DATA_DIR / "task1B_train_cleaned_augmented.jsonl", lines=True, encoding="utf-8")
df_val   = pd.read_json(DATA_DIR / "task1B_dev_cleaned.jsonl", lines=True, encoding="utf-8")
df_test  = pd.read_json(DATA_DIR / "task1B_test_cleaned.jsonl", lines=True, encoding="utf-8")


In [2]:
def compute_vocab_oov(train_texts, val_texts, test_texts):
    """
    Calcule la différence de vocabulaire (OOV) entre Train et Val, et entre Train et Test.
    """
    # Extraction des vocabulaires uniques
    vocab_train = set(w for text in train_texts if isinstance(text, str) for w in text.split())
    vocab_val   = set(w for text in val_texts if isinstance(text, str) for w in text.split())
    vocab_test  = set(w for text in test_texts if isinstance(text, str) for w in text.split())

    # Mots présents dans Val / Test mais absents du Train
    diff_val  = vocab_val - vocab_train
    diff_test = vocab_test - vocab_train

    return {
        "Train vs Val": {
            "total_vocab_val": len(vocab_val),
            "oov_count": len(diff_val),
            "oov_rate_%": round((len(diff_val) / len(vocab_val)) * 100, 2) if vocab_val else 0.0,
            "oov_words": diff_val
        },
        "Train vs Test": {
            "total_vocab_test": len(vocab_test),
            "oov_count": len(diff_test),
            "oov_rate_%": round((len(diff_test) / len(vocab_test)) * 100, 2) if vocab_test else 0.0,
            "oov_words": diff_test
        }
    }

In [3]:
results = compute_vocab_oov(df_train["text"], df_val["text"], df_test["text"])

# Affichage des résultats
print(f"OOV Train vs Val  : {results['Train vs Val']['oov_count']} mots ({results['Train vs Val']['oov_rate_%']}%)")
print(f"OOV Train vs Test : {results['Train vs Test']['oov_count']} mots ({results['Train vs Test']['oov_rate_%']}%)")

OOV Train vs Val  : 1677 mots (35.82%)
OOV Train vs Test : 3154 mots (43.55%)


# Tokenize BASIC

FastText constate que le mot exact n'est pas dans son dictionnaire.
Il découpe immédiatement ce mot inconnu en sous-mots : '<سي', 'سيك', 'يكت', 'كتب', 'تبو', 'بون', 'ون>'...
Il retrouve les vecteurs de ces fragments (qu'il connaît déjà grâce à d'autres mots comme سيارة, يكتب, يلعبون, etc.).
Il fait simplement la somme (ou la moyenne) des vecteurs de ses fragments
v
mot_inconnu
	​

=
g∈subwords
∑
	​

v
g
	​


Le vecteur ainsi créé n'est pas aléatoire : il hérite du sens sémantique de sa racine (كتب) et de ses préfixes/suffixes (سيـ, ـون).

In [4]:
def tokenize(text: str) -> list[str]:
    """Découpage basique sur les espaces pour texte déjà nettoyé."""
    return text.split() if isinstance(text, str) else []

In [5]:
for name, df in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
    # Ajout de la liste des tokens
    df["tokens"] = df["text"].apply(tokenize)
    # Ajout du nombre de mots par texte
    df["token_count"] = df["tokens"].apply(len)
    print(f"✅ {name:10} tokenisé ({len(df):,} exemples | {df['token_count'].sum():,} tokens au total)")
# 4. Aperçu du résultat sur le Train
print("\n--- Exemple de texte tokenisé (Train) ---")
display(df_train[["text", "tokens", "token_count"]].head(3))

✅ Train      tokenisé (2,535 exemples | 71,335 tokens au total)
✅ Validation tokenisé (259 exemples | 8,164 tokens au total)
✅ Test       tokenisé (503 exemples | 14,751 tokens au total)

--- Exemple de texte tokenisé (Train) ---


,text,tokens,token_count
0,جوجل يحتفل بعيد استقلال المملكة الأردنية الهاش...,"[جوجل, يحتفل, بعيد, استقلال, المملكة, الأردنية...",11
1,عاجل الأمن العراقي الهجوم تسبب باستشهاد آمر ال...,"[عاجل, الأمن, العراقي, الهجوم, تسبب, باستشهاد,...",19
2,وبحسب المنظمة الحقوقية الدولية، فقد دعت النياب...,"[وبحسب, المنظمة, الحقوقية, الدولية،, فقد, دعت,...",29


# charger le model

In [6]:
from huggingface_hub import hf_hub_download
import fasttext

model = fasttext.load_model(
    hf_hub_download("facebook/fasttext-ar-vectors", "model.bin")
)   

c:\Users\user\Projets_Ghoufrane\LLMS\Decetction-Persuation\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.bin: reconstructing file:   0%|          |  0.00B / 7.25GB            

model.bin: downloading bytes:           |  0.00B            

KeyboardInterrupt: 

In [ ]:
# 1. Vocabulaire unique du Train
vocab_train = set(w for text in df_train["text"] for w in text.split())

# 2. Vocabulaire du modèle (AraVec ou FastText chargé via Gensim)
vocab_model = set(wv.key_to_index.keys())

# 3. Mots du Train absents du modèle (OOV)
oov_train = vocab_train - vocab_model

# 4. Statistiques
oov_count = len(oov_train)
oov_rate = (oov_count / len(vocab_train)) * 100 if vocab_train else 0.0

print(f"Taille vocabulaire Train   : {len(vocab_train):,} mots")
print(f"Mots OOV (hors modèle)     : {oov_count:,} mots ({oov_rate:.2f}%)")
print(f"Couverture du modèle       : {100 - oov_rate:.2f}%")

    """
    Transforme une collection de textes en une matrice 2D d'embeddings moyens (Mean Pooling).
    
    Logique :
    - Détecte automatiquement la dimension du modèle (wv.vector_size).
    - Calcule le vecteur neutre global (centre de gravité sémantique du modèle)
      pour remplacer les textes vides ou sans mot valide (évite les zéros).
    - Fait la moyenne des vecteurs pour chaque mot du texte.
    
    Args:
        texts: Liste ou pandas Series de textes (ex: df["text"]).
        wv: Objet de vecteurs Gensim (AraVec ou FastText, ex: model.wv).
        show_progress: Affiche une barre de progression tqdm.
        
    Returns:
        np.ndarray de forme (N_textes, embedding_dim) en float32.
    """

Mean Pooling : effectue la moyenne point par point de tous les mots du texte pour résumer la phrase entière en un seul vecteur représentatif de 300 dimensions.

Empile verticalement tous les vecteurs individuels pour créer une matrice 2D exploitable directement par vos classifieurs ou modèles de Deep Learning (shape = (nb_lignes, 300)).

In [ ]:
import numpy as np
from tqdm.auto import tqdm

def embed_corpus(texts, wv, show_progress: bool = True) -> np.ndarray:
    dim = wv.vector_size
    
    # 1. Calcul du vecteur neutre moyen du modèle (centre de gravité sémantique)
    # Si le modèle possède une matrice 'vectors' (Gensim classique) :
    if hasattr(wv, "vectors") and len(wv.vectors) > 0:
        neutral_vector = np.mean(wv.vectors, axis=0).astype(np.float32)
    else:
        # Secours aléatoire centré si les vecteurs bruts ne sont pas exposés
        neutral_vector = np.random.normal(loc=0.0, scale=0.01, size=(dim,)).astype(np.float32)

    def embed_single_text(text: str) -> np.ndarray:
        if not isinstance(text, str) or not text.strip():
            return neutral_vector

        tokens = text.split()
        vectors = []
        for word in tokens:
            try:
                vectors.append(wv[word])
            except KeyError:
                continue

        # Si aucun mot valide trouvé dans le texte -> vecteur neutre (pas de zéros)
        if len(vectors) == 0:
            return neutral_vector

        # Moyenne des vecteurs des mots
        return np.mean(vectors, axis=0).astype(np.float32)

    # 2. Vectorisation de tous les textes
    iterator = tqdm(texts, desc="Vectorisation") if show_progress else texts
    embeddings = [embed_single_text(t) for t in iterator]

    return np.vstack(embeddings)